In [2]:
# Cell 1 - imports & device
#%pip install transformers -q  # uncomment if you don't have transformers installed

import os
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import classification_report, f1_score, precision_score
from tqdm import tqdm
import joblib

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [3]:
# Cell 2 - config
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

FAST_DEBUG = True          # True -> use a SUBSET for quick runs. Set False to train on all data.
SUBSET_SIZE = 50000        # used when FAST_DEBUG = True
EPOCHS = 3                 # epochs for FAST_DEBUG; set higher if FAST_DEBUG=False (e.g., 2-3)
BATCH_SIZE = 16            # reduce if OOM
MODEL_NAME = "unitary/toxic-bert"
MAX_LEN = 128
LR_HEAD = 2e-4             # relatively high LR for head-only tuning
LR_BASE = 1e-5             # tiny if we unfreeze some base layers
PATIENCE = 2               # early stopping patience on validation macro-F1
MIN_PRECISION = 0.03       # min precision threshold for threshold search fallback
print("FAST_DEBUG:", FAST_DEBUG, "SUBSET_SIZE:", SUBSET_SIZE, "EPOCHS:", EPOCHS)


FAST_DEBUG: True SUBSET_SIZE: 50000 EPOCHS: 3


In [4]:
# Cell 3 - load data
df = pd.read_csv("../data/processed_text.csv", keep_default_na=False)

# prefer raw comment_text if present else processed_text
if "comment_text" in df.columns and df["comment_text"].str.len().sum() > 0:
    text_col = "comment_text"
else:
    text_col = "processed_text"

texts_all = df[text_col].astype(str).tolist()
y_train = np.load("../data/y_train.npy").astype(np.float32)
y_test  = np.load("../data/y_test.npy").astype(np.float32)
label_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("Using text column:", text_col)
print("Total texts:", len(texts_all))
print("y_train shape:", y_train.shape, "y_test shape:", y_test.shape)


Using text column: comment_text
Total texts: 159571
y_train shape: (127656, 6) y_test shape: (31915, 6)


In [5]:
# Cell 4 - train/val split (same approach you've used)
from sklearn.model_selection import train_test_split

texts_train_val = texts_all[:len(y_train)]
X_train_texts, X_val_texts, y_train_labels, y_val_labels = train_test_split(
    texts_train_val, y_train, test_size=0.1, random_state=SEED, shuffle=True
)

print("Train samples:", len(X_train_texts))
print("Val samples:", len(X_val_texts))
print("Positive counts (train):", y_train_labels.sum(axis=0))


Train samples: 114890
Val samples: 12766
Positive counts (train): [10976.  1131.  6034.   371.  5583.   995.]


In [6]:
# Cell 5 - tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Attempt to load model as-is (do NOT pass num_labels so pre-finetuned head is kept if present)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
# If the loaded model has different num_labels than our labels, replace classifier weights carefully:
if getattr(model.config, "num_labels", None) != len(label_names):
    # re-init classifier to match our label count (this may reset head)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_names))

model.to(device)
print("Loaded model:", MODEL_NAME)
print("Model num_labels:", model.config.num_labels)


Loaded model: unitary/toxic-bert
Model num_labels: 6


In [7]:
# Cell 6 - tokenization and dataset
def tokenize_texts(texts, max_len=MAX_LEN):
    return tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

class ToxicDataset(Dataset):
    def __init__(self, texts, labels, max_len=MAX_LEN):
        self.enc = tokenize_texts(texts, max_len=max_len)
        self.labels = torch.tensor(labels, dtype=torch.float32)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.enc.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = ToxicDataset(X_train_texts, y_train_labels, max_len=MAX_LEN)
val_dataset   = ToxicDataset(X_val_texts, y_val_labels, max_len=MAX_LEN)
test_texts = texts_all[len(y_train):]
test_dataset = ToxicDataset(test_texts, y_test, max_len=MAX_LEN)

print("Datasets sizes train/val/test:", len(train_dataset), len(val_dataset), len(test_dataset))


Datasets sizes train/val/test: 114890 12766 31915


In [8]:
# Cell 7 - weighted sampler and dataloaders
label_counts = y_train_labels.sum(axis=0)
n_samples = len(y_train_labels)

# compute per-label rarity weight
label_weights = (n_samples - label_counts + 1.0) / (label_counts + 1e-12)

# per-sample weight = mean label weight across positive labels, fallback 1.0
sample_weights = []
for row in y_train_labels:
    positives = row.astype(bool)
    if positives.sum() == 0:
        sample_weights.append(1.0)
    else:
        sample_weights.append(float(label_weights[positives].mean()))
sample_weights = np.array(sample_weights)

if FAST_DEBUG:
    subset_indices = np.random.choice(np.arange(n_samples), min(SUBSET_SIZE, n_samples), replace=False)
    train_subset = Subset(train_dataset, subset_indices)
    sw = sample_weights[subset_indices]
else:
    train_subset = train_dataset
    sw = sample_weights

sampler = WeightedRandomSampler(weights=sw, num_samples=len(sw), replacement=True)
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("Train batches:", len(train_loader), "Val batches:", len(val_loader))


Train batches: 3125 Val batches: 798


In [9]:
# Cell 8 - freeze backbone (fast & safe), train classifier head
# Freeze all base parameters first
for n, p in model.named_parameters():
    p.requires_grad = False

# Identify classifier / head parameters and unfreeze them
head_param_names = []
for n, p in model.named_parameters():
    if "classifier" in n or "pre_classifier" in n or "pre_classifier" in n:
        p.requires_grad = True
        head_param_names.append(n)
# If model uses a different head naming, also allow all parameters in final layers to be trainable
# Print a sample of trainable params
trainable = [n for n, p in model.named_parameters() if p.requires_grad]
print("Trainable params (head):", trainable)

# Optimizer: head-only LR
optimizer = AdamW([p for n,p in model.named_parameters() if p.requires_grad], lr=LR_HEAD, weight_decay=0.01)

num_training_steps = EPOCHS * len(train_loader)
num_warmup_steps = max(1, int(0.06 * num_training_steps))
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))
print("Optimizer prepared. Steps:", num_training_steps)


Trainable params (head): ['classifier.weight', 'classifier.bias']
Optimizer prepared. Steps: 9375


C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\4143357930.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type=="cuda"))


In [10]:
# Cell 9 - focal loss and predict helper
def focal_loss_with_logits(logits, targets, alpha=0.25, gamma=2.0):
    probs = torch.sigmoid(logits)
    ce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p_t = probs * targets + (1 - probs) * (1 - targets)
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    loss = alpha_t * (1 - p_t) ** gamma * ce_loss
    return loss.mean()

def predict_probs(model, dataloader):
    model.eval()
    all_probs = []
    all_labels = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
                logits = model(input_ids, attention_mask=attention_mask).logits
                probs = torch.sigmoid(logits)
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())
    return torch.vstack(all_probs).numpy(), torch.vstack(all_labels).numpy()


In [11]:
# Cell 10 - training loop
best_val = -1.0
best_state = None
stalled = 0

def optimize_thresholds_with_min_prec(y_true, y_pred_probs, pmin=MIN_PRECISION, grid=np.linspace(0,1,101)):
    best_thresholds = []
    for j in range(y_true.shape[1]):
        best_f = -1.0
        best_t = 0.5
        for t in grid:
            preds = (y_pred_probs[:, j] >= t).astype(int)
            prec = precision_score(y_true[:, j], preds, zero_division=0)
            f1 = f1_score(y_true[:, j], preds, zero_division=0)
            if prec >= pmin and f1 > best_f:
                best_f = f1
                best_t = t
        if best_f < 0:
            # fallback pure F1
            best_f = -1.0
            for t in grid:
                preds = (y_pred_probs[:, j] >= t).astype(int)
                f1 = f1_score(y_true[:, j], preds, zero_division=0)
                if f1 > best_f:
                    best_f = f1
                    best_t = t
        best_thresholds.append(best_t)
    return np.array(best_thresholds)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    t0 = time.time()
    loop = tqdm(train_loader, desc=f"Train epoch {epoch+1}/{EPOCHS}")
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
            logits = model(input_ids, attention_mask=attention_mask).logits
            loss = focal_loss_with_logits(logits, labels, alpha=0.25, gamma=2.0)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    dur = time.time() - t0
    avg_loss = total_loss / len(train_loader)

    # Validation
    val_probs, y_val_np = predict_probs(model, val_loader)
    val_preds_half = (val_probs >= 0.5).astype(int)
    val_macro_f1_half = f1_score(y_val_np, val_preds_half, average="macro", zero_division=0)

    # threshold search with min precision
    cur_thresholds = optimize_thresholds_with_min_prec(y_val_np, val_probs, pmin=MIN_PRECISION)
    val_preds_opt = (val_probs >= cur_thresholds).astype(int)
    val_macro_f1_opt = f1_score(y_val_np, val_preds_opt, average="macro", zero_division=0)

    print(f"\nEpoch {epoch+1} — avg_loss: {avg_loss:.4f} — val_macro_f1@0.5: {val_macro_f1_half:.4f} — val_macro_f1_opt: {val_macro_f1_opt:.4f} — time: {dur:.1f}s")
    print("cur thresholds:", np.round(cur_thresholds,3))

    # early stopping on optimized macro F1
    if val_macro_f1_opt > best_val:
        best_val = val_macro_f1_opt
        best_state = {k:v.cpu() for k,v in model.state_dict().items()}
        stalled = 0
        print("New best model saved (in-memory).")
    else:
        stalled += 1
        print(f"No improvement (stalled {stalled}/{PATIENCE}).")
        if stalled >= PATIENCE:
            print("Early stopping triggered.")
            break

# load best state
if best_state is not None:
    model.load_state_dict(best_state)
    model.to(device)
    print("Loaded best model state. Best val macro F1 (opt):", best_val)
else:
    print("No best state saved; using current model.")


Train epoch 1/3:   0%|          | 0/3125 [00:00<?, ?it/s]C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3401317188.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
Train epoch 1/3: 100%|██████████| 3125/3125 [03:36<00:00, 14.44it/s]
C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3305364726.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):



Epoch 1 — avg_loss: 0.0889 — val_macro_f1@0.5: 0.0031 — val_macro_f1_opt: 0.0727 — time: 216.5s
cur thresholds: [0.42 0.32 0.   0.26 0.37 0.18]
New best model saved (in-memory).


Train epoch 2/3:   0%|          | 0/3125 [00:00<?, ?it/s]C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3401317188.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
Train epoch 2/3: 100%|██████████| 3125/3125 [03:09<00:00, 16.48it/s]
C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3305364726.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):



Epoch 2 — avg_loss: 0.0554 — val_macro_f1@0.5: 0.0010 — val_macro_f1_opt: 0.0722 — time: 189.6s
cur thresholds: [0.42 0.33 0.34 0.26 0.33 0.28]
No improvement (stalled 1/2).


Train epoch 3/3:   0%|          | 0/3125 [00:00<?, ?it/s]C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3401317188.py:40: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):
Train epoch 3/3: 100%|██████████| 3125/3125 [02:53<00:00, 17.98it/s]
C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3305364726.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):



Epoch 3 — avg_loss: 0.0553 — val_macro_f1@0.5: 0.0013 — val_macro_f1_opt: 0.0731 — time: 173.8s
cur thresholds: [0.44 0.21 0.35 0.27 0.35 0.31]
New best model saved (in-memory).
Loaded best model state. Best val macro F1 (opt): 0.07307393039167619


In [12]:
# Cell 11 - final thresholds and validation report
val_probs, y_val_np = predict_probs(model, val_loader)
final_thresholds = optimize_thresholds_with_min_prec(y_val_np, val_probs, pmin=MIN_PRECISION)
print("Final thresholds per label:")
for nm, t in zip(label_names, final_thresholds):
    print(f"{nm:15s} -> {t:.3f}")

val_preds_final = (val_probs >= final_thresholds).astype(int)
print("\nValidation classification report:")
print(classification_report(y_val_np, val_preds_final, target_names=label_names, digits=4))
print("Validation macro F1:", f1_score(y_val_np, val_preds_final, average='macro', zero_division=0))


C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3305364726.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):


Final thresholds per label:
toxic           -> 0.440
severe_toxic    -> 0.210
obscene         -> 0.350
threat          -> 0.270
insult          -> 0.350
identity_hate   -> 0.310

Validation classification report:
               precision    recall  f1-score   support

        toxic     0.1006    0.9707    0.1823      1262
 severe_toxic     0.0113    0.9930    0.0223       143
      obscene     0.0555    0.9614    0.1049       700
       threat     0.0048    0.0909    0.0091        33
       insult     0.0545    0.9676    0.1032       680
identity_hate     0.2500    0.0086    0.0167       116

    micro avg     0.0545    0.9209    0.1029      2934
    macro avg     0.0794    0.6654    0.0731      2934
 weighted avg     0.0796    0.9209    0.1292      2934
  samples avg     0.0539    0.0979    0.0651      2934

Validation macro F1: 0.07307393039167619


c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

In [13]:
# Cell 12 - test evaluation
test_probs, y_test_np = predict_probs(model, test_loader)
test_preds = (test_probs >= final_thresholds).astype(int)
print("\nTest classification report (optimized thresholds):")
print(classification_report(y_test_np, test_preds, target_names=label_names, digits=4))
print("Test macro F1:", f1_score(y_test_np, test_preds, average='macro', zero_division=0))


C:\Users\Yasna\AppData\Local\Temp\ipykernel_24004\3305364726.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type=="cuda")):



Test classification report (optimized thresholds):
               precision    recall  f1-score   support

        toxic     0.0959    0.9539    0.1742      3056
 severe_toxic     0.0099    0.9751    0.0197       321
      obscene     0.0536    0.9475    0.1015      1715
       threat     0.0020    0.0405    0.0039        74
       insult     0.0514    0.9659    0.0976      1614
identity_hate     0.1579    0.0102    0.0192       294

    micro avg     0.0518    0.9073    0.0979      7074
    macro avg     0.0618    0.6489    0.0693      7074
 weighted avg     0.0732    0.9073    0.1239      7074
  samples avg     0.0515    0.0936    0.0623      7074

Test macro F1: 0.06934473678066631


c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} 

In [14]:
# Cell 13 - save artifacts
os.makedirs("artifacts", exist_ok=True)
torch.save(model.state_dict(), "artifacts/unitary_toxic_bert_head_only.pth")
joblib.dump(final_thresholds, "artifacts/final_thresholds.npy")
print("Saved model state and thresholds to artifacts/")


Saved model state and thresholds to artifacts/
